In [ ]:
!pip install -U langgraph langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [ ]:
from typing import Annotated, TypedDict
import operator

class AgentState(TypedDict):
  messages : Annotated[list , operator.add]

In [ ]:
import os
import json
from typing import Annotated, TypedDict
from google.colab import userdata # For your API Key

# LangChain / Groq Imports
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# LangGraph Core Imports
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Initialize your LLM (using the stable 2026 model)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=userdata.get('RealEst')
)

def classNode(state : AgentState):

  text = state['messages'][-1].content
  response = llm.invoke(text)
  return {"messages": [response]}


In [ ]:
def should_continue(state : AgentState):
  messages = state['messages']
  lastM = messages[-1].content

  if lastM == "formal":
    return "formal_worker"
  elif lastM == "casual":
    return "casual_worker"
  else:
    return "end"


In [ ]:
def formal_worker(state : AgentState):
  sys_msg = SystemMessage(content="You are a corporate assistant. Take the rough note in the history and write a professional email. Return ONLY the email body.")
  messages = [sys_msg] + state["messages"]

  response = llm.invoke(messages)
  return {"messages": [response]}



def casual_worker(state : AgentState):
  sys_msg = SystemMessage(content="""
        You are a friendly, laid-back friend.
        Take the rough note in the history and turn it into a warm, casual message.
        Use emojis, abbreviations (like 'u' or 'nvm'), and a friendly tone.
        Return ONLY the message body.
    """)

    # 2. Inject context
  messages = [sys_msg] + state["messages"]

  response = llm.invoke(messages)
  return {"messages": [response]}



In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)
builder.add_node("classifier",classNode)
builder.add_node("formal_node", formal_worker)
builder.add_node("casual_node", casual_worker)
builder.add_edge(START, "classifier")
builder.add_edge("formal_node", END)
builder.add_edge("casual_node", END)
builder.add_conditional_edges(
    "classifier",
    should_continue,
    {
        "formal_worker": "formal_node",
        "casual_worker": "casual_node",
        "end": END
    }
)

In [ ]:
app = builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage
inputs = {
    "messages": [HumanMessage(content="Rough note: tell the team the meeting is moved to 4pm")]
}

final_state = app.invoke(inputs,config={"recursion_limit": 5})

# 3. Access the "Response"
# The 'final_state' is just a dictionary containing the updated Baton
print(final_state["messages"][-1].content)

Meeting update: the meeting has been rescheduled to 4pm.
